Clustering Routine for Skills Project:Phase 2<br>
Written by: Robert Methven (@methvenr)<br>
Date: April 2025<br>

In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('z_scores_scaled/combined_z_scores.csv', delimiter=',')

# Get list of columns with 'percentile' in the name
percentile_cols = [col for col in df.columns if 'percentile' in col.lower()]

# Create new dataframe with emplid and percentile columns
selected_df = df[['emplid','segment','cluster_name'] + percentile_cols]

# Display first few rows
print(selected_df.head())

# If you want to see all column names
print("\nSelected columns:")
print(selected_df.columns.tolist())

# If you want to save to a new CSV file
# selected_df.to_csv('percentile_scores.csv', index=False)


In [8]:
result_df = selected_df.melt(
        id_vars=['emplid','segment','cluster_name'],
        var_name='field_name',
        value_name='score'
        )

In [9]:
import pandas as pd

# Filter for scores - Change the threshold below
# focuses recommendations on low-performing areas
low_scores = result_df[result_df['score'] < 71].copy()

# Initialize lists to store results
original_emplids = []
field_names = []
original_scores = []
top_emplids = []
top_scores = []
top_cluster_names = []
segments = []
source_groups = []  # To track if recommendation came from cluster or segment

def get_top_recommendations(row, result_df, n_recommendations=5):
    """
    This function finds top recommendations either from the same cluster or segment.
    Benefits:
    - Prioritizes recommendations from the same cluster for more relevant matches
    - Falls back to segment recommendations if no cluster matches are found
    - Allows for flexible number of recommendations
    """
    # First get recommendations from the same cluster - ensures we're comparing with the most similar peers
    cluster_recommendations = (
        result_df[
            (result_df['field_name'] == row['field_name']) &
            (result_df['cluster_name'] == row['cluster_name']) &
            (result_df['emplid'] != row['emplid'])
        ]
        .nlargest(n_recommendations, 'score')
    )
    
    # No cluster recommendations then use segment. This provides a fallback for broader, but still relevant recommendations
    if cluster_recommendations.empty:
        segment_recommendations = (
            result_df[
                (result_df['field_name'] == row['field_name']) &
                (result_df['segment'] == row['segment']) &
                (result_df['emplid'] != row['emplid'])
            ]
            .nlargest(n_recommendations, 'score')
        )
        
        recommendations = segment_recommendations
        sources = ['segment'] * len(segment_recommendations)
    else:
        recommendations = cluster_recommendations
        sources = ['cluster'] * len(cluster_recommendations)
    
    return recommendations, sources

# Iterate through each low score to find recommendations - ensures we're addressing each area that needs improvement
for idx, row in low_scores.iterrows():
    # Get recommendations
    top_five, rec_sources = get_top_recommendations(row, result_df, 5)
    
    # Collect data for each recommendation - prepares the data for DataFrame creation
    if not top_five.empty:
        for (_, top_row), source in zip(top_five.iterrows(), rec_sources):
            original_emplids.append(row['emplid'])
            field_names.append(row['field_name'])
            original_scores.append(row['score'])
            top_emplids.append(top_row['emplid'])
            top_scores.append(top_row['score'])
            top_cluster_names.append(top_row['cluster_name'])
            segments.append(row['segment'])
            source_groups.append(source)

# Create result DataFrame
recommendation_df = pd.DataFrame({
    'original_emplid': original_emplids,
    'field_name': field_names,
    'original_score': original_scores,
    'top_scoring_emplid': top_emplids,
    'top_score': top_scores,
    'top_cluster_name': top_cluster_names,
    'segment': segments,
    'recommendation_source': source_groups
})

# Sort by original_emplid and top_score (descending)
recommendation_df = recommendation_df.sort_values(['original_emplid', 'top_score'], 
                                                  ascending=[True, False])

# Add rank within each original_emplid and field_name combination
recommendation_df['rank'] = (recommendation_df.groupby(['original_emplid', 'field_name'])
                    .cumcount() + 1)

# Sort by emplid, field_name, and rank
recommendation_df = recommendation_df.sort_values(['original_emplid', 'field_name', 'rank'])

# Reset index
recommendation_df = recommendation_df.reset_index(drop=True)

# Rename column to skill
recommendation_df.rename(columns={'field_name': 'skill'}, inplace=True)

# Remove '_z_score_percentile' from column names
recommendation_df['skill'] = recommendation_df['skill'].str.replace('_z_score_percentile', '')

# Save to CSV file
recommendation_df.to_csv('Recommendations/Peer/top_5_peers_by_cluster_or_segment.csv', index=False)
